# Snowball Strategy

In [ ]:
%reload_ext autoreload
%autoreload 2

from datetime import UTC, datetime, timedelta
from pathlib import Path

from aws import AthenaDataSource
from core import (
    BacktestTaskDefinition,
    CsvResultStore,
    CurrencyPair,
    InMemoryTaskRegistry,
    LogLevel,
    Pips,
    SpreadFilter,
    StrategyParameters,
    TaskManager,
    TaskProfilingConfig,
    TaskResultRecorder,
    TickGranularity,
    TickGranularityFilter,
    TqdmProgressReporter,
    configure_logging,
)
from snowball import SnowballStrategy

from notebooks.backtests import (
    BacktestChartConfig,
    BacktestResultChart,
    SortablePaginatedDataFrameDisplay,
    TaskResultFrameBuilder,
    TaskResultFrameDisplay,
)

configure_logging(level=LogLevel.WARNING)


## Parameters

In [ ]:
INSTRUMENT = CurrencyPair.of("USD_JPY")
START_AT = datetime(2026, 1, 1, tzinfo=UTC)
END_AT = datetime(2026, 7, 12, 23, 59, 59, 999999, tzinfo=UTC)
SAMPLE_GRANULARITY = TickGranularity.TICK
CHART_DAY_AGGS_THRESHOLD_DAYS = 31
PROFILE_DIR = Path("profiles") / "snowball"
RESULT_DIR = Path("results") / "snowball"
LOCAL_TIMEZONE = datetime.now().astimezone().tzinfo

data_source = AthenaDataSource.from_env().with_filters(
    SpreadFilter.of(Pips("5")),
    TickGranularityFilter.of(SAMPLE_GRANULARITY),
)

parameters = SnowballStrategy.normalize_parameters(
    StrategyParameters.of(
        sizing={
            # choices: "fixed", "balance_based"
            "mode": "balance_based",
            "base_units": "1500",
            "balance_based": {
                "round_step_units": "1",
                "min_units": "1",
            },
        },
        grid={
            "max_retracements_per_layer": 6,
            "max_layers": 30,
            "refill": {
                "enabled": True,
            },
        },
        cycle={
            "hedging_enabled": True,
            "reseed_when_all_positions_pending_rebuild": True,
        },
        forward={
            "take_profit_pips": "30",
        },
        counter={
            "interval": {
                # choices: "constant", "additive", "subtractive"
                #          "multiplicative", "divisive", "manual"
                "mode": "divisive",
                "head_pips": "30",
                "tail_pips": "10",
                "gamma": "1.3",
            },
            "take_profit": {"mode": "weighted_avg"},
        },
        stop_loss={
            "enabled": False,
            # choices: "auto", "distance"
            "mode": "auto",
        },
        rebuild={
            "enabled": True,
            "price": {
                # choices: "original_entry_price", "stop_loss_exit_price"
                "entry_price_mode": "original_entry_price",
            },
            "stop_loss": {
                # choices: "same_price", "same_distance", "manual_distance"
                "mode": "same_distance",
            },
            "take_profit": {
                # choices: "same_price", "same_distance", "progressive_distance"
                "mode": "same_distance"
            },
        },
        protection={
            "shrink_enabled": False,
            "shrink_start_margin_percent": "70",
            "shrink_target_margin_percent": "50",
            "emergency_enabled": True,
            "emergency_margin_percent": "95",
        },
        account={
            "initial_balance": {
                "amount": "3000000",
                "currency": "JPY",
            },
            "margin_rate": "0.04",
            "quote_to_account_rate": "1",
        },
    )
)


## Executuin

In [ ]:
definition = BacktestTaskDefinition(
    name=f"{INSTRUMENT} Snowball backtest",
    instrument=INSTRUMENT,
    start_at=START_AT,
    end_at=END_AT,
    parameters=parameters,
)

strategy = SnowballStrategy(
    name="snowball",
    parameters=definition.parameters,
)

recorder = TaskResultRecorder(
    stores=(CsvResultStore(RESULT_DIR),),
    metric_interval=timedelta(minutes=5),
)

with TaskManager(
    registry=InMemoryTaskRegistry(),
    observers=(recorder,),
    event_handlers=(recorder,),
    profiling=TaskProfilingConfig.for_backtest_period(
        start_at=START_AT,
        end_at=END_AT,
        cprofile=False,
        cprofile_output_path=PROFILE_DIR,
        pyinstrument=False,
        pyinstrument_output_path=PROFILE_DIR,
    ),
) as manager:
    run = manager.start_backtest(definition, data_source=data_source, strategy=strategy)
    final_task = run.wait(progress=TqdmProgressReporter())

if final_task.failure is not None:
    print(f"Task failed: {final_task.failure.message}")
    print(f"Cause: {final_task.failure.cause_type}")
    print(final_task.failure.traceback)
    raise RuntimeError(str(final_task.failure))

print(f"Events recorded: {len(recorder.event_records(final_task.id))}")
print(f"Trades recorded: {len(recorder.trade_summaries(final_task.id))}")


## Results

In [ ]:
frames = TaskResultFrameBuilder(timezone=LOCAL_TIMEZONE).from_recorder(
    recorder=recorder,
    task=final_task,
)

events_df = frames.events
signals_df = frames.signals
trade_summary_df = frames.trades
cycle_summary_df = frames.cycles
task_summary_df = frames.task
metrics_df = frames.metrics

TaskResultFrameDisplay(
    table_display=SortablePaginatedDataFrameDisplay(page_size=100),
).display(frames)


## Chart


In [ ]:
chart_config = BacktestChartConfig(
    instrument=INSTRUMENT,
    start_at=START_AT,
    end_at=END_AT,
    local_timezone=LOCAL_TIMEZONE,
    day_aggs_threshold_days=CHART_DAY_AGGS_THRESHOLD_DAYS,
)
chart = BacktestResultChart(data_source=data_source, config=chart_config)
chart_granularity = chart.granularity
candles = chart.candles()
candles_df = chart.candle_frame(candles)
chart.display(frames=frames, candles=candles_df)


In [ ]:
profile = run.profile()
profile_outputs = {
    "cprofile": profile.cprofile_output_path,
    "pyinstrument": profile.pyinstrument_output_path,
}
if profile.cprofile_output_path is not None:
    profile_outputs["snakeviz"] = f"uv run snakeviz {profile.cprofile_output_path}"
    profile_outputs["tuna"] = f"uv run tuna {profile.cprofile_output_path}"
if profile.pyinstrument_output_path is not None:
    profile_outputs["html"] = f"open {profile.pyinstrument_output_path}"
profile_outputs